## Transfer Learning 

- 이전에 학습 된 모델을 새로운 문제에 적용하는학습 방법
- 이미 학습된 모델을 유사한 문제에 적용 
- 한 작업에서 얻은 학습 내용을 다른 작업에 전이, 새로운 작업에서의 성능을 향상 시키는 것 

- 이미 사전 학습 된 모델은 대규모 데이터셋(예, ImageNet )에서 학습 된 가중치를 가지고 있으며, 이를 그대로 사용하거나, 새로운 데이터셋에 맞춰 미세 조정(fine tuning)하여 다시 학습할 수 있다. 
- 모델의 가중치를 초기화하지 않고 유지, 새로운 데이터셋에 맞는 마지막 레이어만 학습하거나 일부 레이어만 재학습. 

In [2]:
import torch
import torchvision.models as models

# ResNet18 모델을 사전 학습된 가중치와 함께 불러오기
model = models.resnet18(pretrained=True)

# 모델 구조 확인
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 데이터셋 전처리
transform = transforms.Compose([
    transforms.Resize(224),  # ResNet 입력 크기 맞추기
    transforms.ToTensor(),  # 텐서로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 사전 학습된 모델에 맞는 정규화
])

# ImageFolder를 사용하여 train 데이터셋 불러오기
train_dataset = ImageFolder(root='./train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 학습 루프 간단 구현
for inputs, labels in train_loader:
    outputs = model(inputs)
    print(outputs)  # 예측 결과 출력
    print(f'\n출력 데이터 크기:', outputs.shape)
    print(f'\n32개 이미지에 대한 예측된 클래스:', outputs.argmax(axis=1))
    break  # 한 배치만 확인

FileNotFoundError: [Errno 2] No such file or directory: './train'

```
tensor([[ 0.9821, -1.5525,  1.0293,  ..., -1.4344, -0.2788,  0.5903],
        [ 1.7452, -0.0568, -0.8030,  ...,  2.2315,  1.8798, -1.1263],
        [-2.2867, -2.5841, -0.5504,  ...,  0.1290, -0.0725,  1.2060],
        ...,
        [ 0.5681, -1.0880,  0.8877,  ...,  3.3779,  3.2365, -0.2004],
        [ 1.3243,  1.7805,  1.6069,  ..., -0.2049,  0.9064,  2.5498],
        [ 0.1707,  0.3540,  4.7256,  ..., -4.7745, -0.7109,  2.0515]],
       grad_fn=<AddmmBackward0>)

출력 데이터 크기: torch.Size([32, 1000])

32개 이미지에 대한 예측된 클래스: tensor([660, 373, 536, 842, 727, 106, 703, 586, 884, 696, 678,  97, 391, 588,
        783, 674, 942, 657, 891,  11, 805, 996, 902, 491, 987, 261, 470, 592,
        206, 319, 107, 150])
```

## ResNet(Residual Network)

- [잔차 연결](./../06.PretrainedModel/01.Residual_connections.md)을 도입하여, 네트워크가 매우 깊어 질 때 발생하는 기울기 소실 문제를 해결 
    - 이 구조는 레이어간 직접 연결을 추가하여, 네트워크가 깊어져도 성능이 향상 될 수 있도록 돕는다. 
- 깊은 네트워크에서도 성능이 안정적, 다양한 이미지 분류 및 인식 작업에 널리 사용 

## VGG(Visual Geometry Group)

- 여러개의 3x3필터를 쌓아 깊이 있는 CNN(Convolutional Neral Network)을 형성한 모델
- 해석하기 쉽다. 
- 컨볼루션(Convolution Layer) -> 풀링 레이어(Pooling layer) -> 완전 연결 레이어(fully connected layer) -> 소프트맥스 레이어(Softmax layer)

In [4]:
import torch
import torchvision.models as models

# ResNet18 사전 학습된 모델 불러오기
model = models.resnet18(pretrained=True)

# 모델을 평가 모드로 설정 (추론 시 필요)
model.eval()

# 모델 구조 확인
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

```python
import torchvision.transforms as transforms
from PIL import Image

# CIFAR-10 이미지 전처리
transform = transforms.Compose([
    transforms.Resize(224), #CIFAR-10 이미지 크기 32x32 -> ResNet18모델 입력크기는 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 테스트 이미지를 불러오기 (예시로 하나의 이미지 사용)
img = Image.open('cifar_image.jpg')
img_t = transform(img)
batch_t = torch.unsqueeze(img_t, 0)  # 배치 형태로 변환
print(f'배치사이즈: ',batch_t.shape)

# 모델을 사용해 예측
with torch.no_grad():  # 추론 중에는 그래디언트를 계산하지 않음
    out = model(batch_t)

# 결과 출력
print(out)
_, predicted_class = torch.max(out, 1)
print(f"Predicted class: {predicted_class.item()}")
```

## 결과 해석 

- 배치 사이즈: 입력 된 이미지가 1개의 이미지로 구성된 배치, 각각의 이미지가 3개의 색상 채널 (RGB)와 224x224픽셀 크기를 가진다는 것을 의미 
- 출력 텐서 : 출력 된 값들은 로짓(logits)으로, 모델이 각 클래스에 대해 예측한 "점수", 이 점수는 음수나 양수 일 수 있으며, 값이 클수록 해당 클래스에 속할 가능성이 높다는 뜻, ImageNet 데이터셋에서 학습된 ResNet18 모델이므로, 1000개의 ImageNet클래스 중 하나를 예측하게 됩니다. 

- Predict Clas: 335: torch.max(out, 1) 을 사용해 1000개의 클래스 중 가장 높은 점수를 받은 클래스를 예측, 그 결과 `335`번 클래스가 선택 됨. ImageNet 데이터 셋에서 사전 학습된 모델의 클래스 인덱스를 의미 CIFAR-10 클래스와는 다름. 
- `ImageNetdml 335클래스가 CIFAR-10이미지를 보고 가능성이 높다고 판단된 클래스` 

```
배치사이즈:  torch.Size([1, 3, 224, 224])
tensor([[ 4.0363e+00,  1.4217e+00, -1.4910e+00, -4.6077e+00, -3.7689e+00,
          2.1729e+00, -3.9440e+00,  3.7153e-01,  1.1757e+00, -8.5295e-01,
          4.3050e+00, -1.6868e+00, -2.3042e+00, -3.8390e+00, -3.3013e+00,
         -3.3915e+00, -3.2870e+00, -3.5190e+00, -5.4709e+00, -4.4180e+00,
         -1.0408e+00,  1.0697e+00, -2.7281e+00, -1.4952e+00, -1.8120e+00,
          1.1644e+00,  3.8687e+00,  2.7678e+00,  3.6982e+00,  1.2574e+00,
          ...
          ...
          ...
          -2.6103e+00,  1.0516e+00, -9.9031e-01, -3.2472e+00, -2.4508e+00,
         -1.5461e+00,  4.8962e-01,  9.0596e-01, -5.2621e-01, -2.7542e+00,
          1.4469e+00, -3.0322e+00, -7.2181e-01,  2.5004e+00,  5.9578e-01,
          2.5014e+00,  2.0930e+00,  3.6783e+00,  1.8409e+00, -3.0624e+00]])
Predicted class: 335
```

## 사전 학습 모델을 활용한 기본 이미지 분류 

```python
import torch
import torch.nn as nn
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

# CIFAR-10 데이터셋 불러오기
transform = transforms.Compose([
    transforms.Resize(224),  # 입력 크기를 ResNet 모델에 맞추기
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root='./train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 사전 학습된 ResNet18 모델을 불러오고 CIFAR-10에 맞게 출력 레이어 수정
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10) 

model.eval()  

# 학습 루프에서 예측 수행
for inputs, labels in train_loader:
    with torch.no_grad():  
        outputs = model(inputs)  
    print(outputs)  
    break   
```

## 결과 해석 

- 출력된 결과는 모델이 각 배치에 대해 CIFAR-10의 10개 클래스에 대해 예측한 로짓(logits) 값입니다. 
- 

```
tensor([[-0.0152, -0.0788,  0.4812,  0.5106, -0.5678,  0.1927,  0.2802, -0.1230,
         -0.5314,  0.0459],
        [-0.9083, -0.1452, -0.1747,  0.1228, -0.3857,  0.5641,  0.3648, -0.2856,
         -0.7403, -1.0223],
        [-0.1251, -0.1639, -0.1007,  1.1067, -1.0362,  0.2851, -0.4113,  0.0754,
         -0.3897,  0.3234],
        [ 0.1661, -0.4323, -0.0657,  0.8099, -0.3302,  0.2585,  0.4583,  0.0758,
         -0.9323, -0.6302],
        [ 0.4421, -0.7657, -0.3361,  0.4012, -0.2645,  0.3685,  0.5388,  0.4013,
         -0.1352, -0.7587],
```

## Final predict class extract 

- 3이 고양이, 5가 개, 9가 트럭 이라 했을 때, 3은 고양이라 예측, 9는 트럭이라 예측

```python
_, predicted_class = torch.max(outputs, 1)
print(f"Predicted class: {predicted_class}")
```

```
Predicted class: tensor([3, 5, 3, 3, 6, 5, 5, 7, 6, 1, 3, 3, 3, 9, 7, 3, 3, 6, 3, 3, 6, 3, 3, 6,
        7, 3, 3, 7, 5, 6, 5, 3])
```

## 예측 결과 시각화 및 정답 비교 

```python
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# CIFAR-10 이미지 전처리
transform = transforms.Compose([
    transforms.Resize(64), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# CIFAR-10 테스트 데이터셋 로드
testset = ImageFolder(root='./train', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=8,
                                         shuffle=False, num_workers=2)

# CIFAR-10 클래스 정의
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 이미지를 시각화하는 함수 정의
def imshow(img):
    img = img / 2 + 0.5  # 정규화 해제
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# 모델을 평가 모드로 설정
model.eval()

# 8장의 이미지를 가져와 예측
dataiter = iter(testloader)
images, labels = next(dataiter)

# 모델을 사용해 예측
with torch.no_grad():
    outputs = model(images)

_, predicted = torch.max(outputs, 1)

# 이미지와 함께 실제 및 예측된 클래스를 시각화
imshow(torchvision.utils.make_grid(images, nrow=4))  # 2x4 형태로 이미지 배치

# 정답과 예측을 비교하여 함께 출력
for i in range(8):
    print(f"Image {i+1}: GroundTruth = {classes[labels[i]]}, Predicted = {classes[predicted[i]]}")
```

```
Image 1: GroundTruth = plane, Predicted = horse
Image 2: GroundTruth = plane, Predicted = dog
Image 3: GroundTruth = plane, Predicted = ship
Image 4: GroundTruth = plane, Predicted = car
Image 5: GroundTruth = plane, Predicted = dog
Image 6: GroundTruth = plane, Predicted = dog
Image 7: GroundTruth = plane, Predicted = truck
Image 8: GroundTruth = plane, Predicted = bird
```